# 02 — Depuración Silver

Objetivo: recibir los parquets de Bronze, aplicar reglas de calidad, y guardar parquets limpios en Silver.

El propósito de este notebook es aplicar reglas de calidad, no explorar: cada decisión tiene una justificación documentada.

## IMPORTS Y CONFIGURACIÓN DE RUTAS

In [8]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


DATA_DIR = PROJECT_ROOT / 'data'
SILVER = DATA_DIR / 'silver'
SILVER.mkdir(parents=True, exist_ok=True)
BRONZE_SNAP = DATA_DIR / 'bronze' / 'snapshots'


print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')

Project root: /Users/aa/Desktop/TFM/Código/TFM-Hosteleria-AI
Data directory: /Users/aa/Desktop/TFM/Código/TFM-Hosteleria-AI/data


## SECCIÓN 0 — Setup

Cargamos los 12 ficheros parquet

In [9]:
df_tickets     = pd.read_parquet(BRONZE_SNAP / 'tickets_raw.parquet')
df_ventas      = pd.read_parquet(BRONZE_SNAP / 'ventas_raw.parquet')
df_reservas    = pd.read_parquet(BRONZE_SNAP / 'reservas_raw.parquet')
df_tips        = pd.read_parquet(BRONZE_SNAP / 'tips_raw.parquet')
df_artic       = pd.read_parquet(BRONZE_SNAP / 'articulos_raw.parquet')
df_depart      = pd.read_parquet(BRONZE_SNAP / 'departamentos_raw.parquet')
df_menu        = pd.read_parquet(BRONZE_SNAP / 'menu_raw.parquet')
df_festivos    = pd.read_parquet(BRONZE_SNAP / 'festivos_raw.parquet')
df_eventos     = pd.read_parquet(BRONZE_SNAP / 'eventos_raw.parquet')
meteo_diaria   = pd.read_parquet(BRONZE_SNAP / 'meteo_diaria_raw.parquet')
meteo_horaria  = pd.read_parquet(BRONZE_SNAP / 'meteo_horaria_raw.parquet')
total_articles = pd.read_parquet(BRONZE_SNAP / 'total_articles_raw.parquet')

## SECCIÓN 1 — Tabla de auditoría inicial

Antes de aplicar ninguna transformación, inspeccionamos el estado de los 12 datasets
tal como llegaron de Bronze. 

El objetivo es tener una fotografía del punto de partida
que sirva como referencia para documentar qué se modificó y por qué en las secciones
siguientes.

Para cada dataset se calculan cinco métricas:
- **filas / columnas**: dimensiones del dataset
- **nulos_pct_max**: porcentaje de nulos de la columna más afectada
- **duplicados_exactos**: filas idénticas en todas las columnas
- **fecha_min / fecha_max**: rango temporal de la columna de fecha operativa principal
  (NaT para datasets sin dimensión temporal propia: catálogos, tips, total_articles)python

In [10]:
def audit_dataset(nombre, df, col_fecha=None):
    """
    Genera un resumen de calidad de un DataFrame.

    Parameters
    ----------
    nombre : str
        Nombre identificador del dataset.
    df : pd.DataFrame
        Dataset a auditar.
    col_fecha : str or None
        Columna de fecha operativa principal. Si es None, fecha_min y fecha_max
        se devuelven como NaT.

    Returns
    -------
    dict
        Métricas de calidad del dataset.
    """
    fecha_min = pd.NaT
    fecha_max = pd.NaT

    if col_fecha is not None and col_fecha in df.columns:
        col = pd.to_datetime(df[col_fecha], errors='coerce')
        fecha_min = col.min()
        fecha_max = col.max()

    return {
        'dataset':            nombre,
        'filas':              len(df),
        'columnas':           len(df.columns),
        'nulos_pct_max':      round(df.isnull().mean().max() * 100, 2),
        'duplicados_exactos': int(df.duplicated().sum()),
        'fecha_min':          fecha_min,
        'fecha_max':          fecha_max,
    }

In [11]:
# Aplicamos la auditoría a los 12 datasets.
# col_fecha apunta a la columna de fecha operativa de cada dataset,
# no a metadatos del informe (report_start, report_end).
# Los datasets sin dimensión temporal propia reciben col_fecha=None.

auditoria = pd.DataFrame([
    audit_dataset('tickets',        df_tickets,       col_fecha='date'),
    audit_dataset('ventas',         df_ventas,        col_fecha='report_start'),
    audit_dataset('reservas',       df_reservas,      col_fecha='reservation_datetime'),
    audit_dataset('tips',           df_tips,          col_fecha=None),   # sin fecha propia: referencia document_id de tickets
    audit_dataset('articulos',      df_artic,         col_fecha=None),   # catálogo estático
    audit_dataset('departamentos',  df_depart,        col_fecha=None),   # catálogo estático
    audit_dataset('menu',           df_menu,          col_fecha=None),   # catálogo estático
    audit_dataset('festivos',       df_festivos,      col_fecha='fecha'),
    audit_dataset('eventos',        df_eventos,       col_fecha='fecha_inicio'),
    audit_dataset('meteo_diaria',   meteo_diaria,     col_fecha='date'),
    audit_dataset('meteo_horaria',  meteo_horaria,    col_fecha='datetime'),
    audit_dataset('total_articles', total_articles,   col_fecha=None),   # agregado global sin granularidad diaria
])

display(auditoria)

,dataset,filas,columnas,nulos_pct_max,duplicados_exactos,fecha_min,fecha_max
0,tickets,6709,11,0.00,0,2025-10-02 00:00:00,2026-07-09 00:00:00
1,ventas,5532,13,0.00,0,2025-10-01 00:00:00,2026-07-06 00:00:00
2,reservas,21341,20,99.99,0,2022-11-10 13:45:00,2026-07-25 14:00:00
3,tips,1697,11,0.00,0,NaT,NaT
4,articulos,405,5,95.56,0,NaT,NaT
5,departamentos,23,4,100.00,0,NaT,NaT
6,menu,405,3,0.00,0,NaT,NaT
7,festivos,24,4,0.00,0,2025-01-01 00:00:00,2026-12-25 00:00:00
8,eventos,78,20,0.00,0,2025-10-10 00:00:00,2026-09-16 00:00:00
9,meteo_diaria,921,9,0.00,0,2024-01-01 00:00:00,2026-07-09 00:00:00


### Conclusiones de la auditoría inicial

- **Sin duplicados en ningún dataset.** Los 12 datasets pasan el check de filas
  idénticas con resultado 0. No hay registros duplicados que eliminar en esta capa.

- **Los catálogos (artículos, departamentos) presentan nulos estructurales.**
  `article_short_name` está vacío en el 95.6% de los artículos y
  `department_short_name` al 100%. Ambas columnas son residuos del sistema TPV
  sin valor para el modelo; se eliminarán en la sección 2.

- **Reservas acumula el mayor problema de calidad.** El 99.99% de nulos en
  `nulos_pct_max` corresponde a la columna `reference`, que está prácticamente
  vacía. Adicionalmente, `entered_by` (85.6% nulos) y `referrer` (70.2% nulos)
  son columnas operativas del TPV sin utilidad para la predicción. Las columnas
  clave —`reservation_datetime`, `status`, `shift`, `people`, `origin`— tienen
  0 nulos. Los problemas específicos de reservas se detallan en la sección 2.3.

- **Tickets, ventas, meteorología, festivos y eventos están en buen estado
  general.** Sin nulos en columnas operativas y sin duplicados. Los problemas
  puntuales detectados (días sin actividad en tickets, una fila con importe
  negativo en ventas, outliers de importe) se abordan en la sección 2.

## SECCIÓN 2 — Depuración por dataset

Para cada dataset el orden es siempre el mismo:

- tipos → nulos → duplicados → rangos/outliers → coherencia de negocio.

Cada decisión de eliminación o transformación queda justificada en el texto que precede al código.

### 2.1 Tickets

Sin nulos ni duplicados. Los problemas son dos:
- 39 días sin actividad, todos ellos lunes salvo excepciones por festivos en
  martes. Patrón operativo confirmado: el establecimiento cierra los lunes.
  No se eliminan filas; se añade una columna `es_dia_cierre` para que el
  modelo no prediga esos días.
- Un ticket con importe 3.821€ muy por encima del percentil 99.9 (958€).
  Se flaGuea con `flag_outlier_importe` para revisión, no se elimina.

In [19]:
df_tickets.document_total.max()

np.float64(3821.65)

In [ ]:
df_tickets_clean = df_tickets.copy()

# 1. Asegurar tipo datetime
df_tickets_clean['report_start'] = pd.to_datetime(df_tickets_clean['report_start'])
df_tickets_clean['report_end'] = pd.to_datetime(df_tickets_clean['report_end'])
df_tickets_clean['report_generated_on'] = pd.to_datetime(df_tickets_clean['report_generated_on'])
df_tickets_clean['date'] = pd.to_datetime(df_tickets_clean['date'])

# 2. Añadir dia_semana (0=lunes, 6=domingo)
df_tickets_clean['dia_semana'] = df_tickets_clean['date'].dt.dayofweek

# 3. es_dia_cierre → dia_semana == 0
df_tickets_clean['es_dia_cierre'] = df_tickets_clean['dia_semana'] == 0

# 4. flag_outlier_importe → document_total > UMBRAL
UMBRAL_OUTLIER = df_tickets_clean['document_total'].quantile(0.999)  # Definimos el umbral como el percentil 99 de los importes
df_tickets_clean['flag_outlier_importe'] = df_tickets_clean['document_total'] > UMBRAL_OUTLIER
"""
    Q1 = df_tickets_clean['document_total'].quantile(0.25)
    Q3 = df_tickets_clean['document_total'].quantile(0.75)
    IQR = Q3 - Q1
    UMBRAL_OUTLIER = Q3 + 3 * IQR # más conservador
"""

print(f"Umbral: {UMBRAL_OUTLIER:.2f}€")
print(f"Tickets flagueados: {df_tickets_clean['flag_outlier_importe'].sum()}")

# Verificación final
print(f"Filas antes: {len(df_tickets)} | después: {len(df_tickets_clean)}")
print(df_tickets_clean[['date','dia_semana','es_dia_cierre','document_total','flag_outlier_importe']].head())

Umbral: 958.14€
Tickets flagueados: 7
Filas antes: 6709 | después: 6709
        date  dia_semana  es_dia_cierre  document_total  flag_outlier_importe
0 2025-10-02           3          False            7.00                 False
1 2025-10-02           3          False           65.40                 False
2 2025-10-02           3          False            8.20                 False
3 2025-10-02           3          False           77.10                 False
4 2025-10-02           3          False           35.05                 False


### 2.2 Ventas semanales

Un registro con `amount <= 0` con `units > 0` es incoherente:
implica precio nulo o negativo. Se elimina.

In [23]:
df_ventas_clean = df_ventas.copy()

# 1. Asegurar tipo datetime
df_ventas_clean['report_start'] = pd.to_datetime(df_ventas_clean['report_start'])
df_ventas_clean['report_end'] = pd.to_datetime(df_ventas_clean['report_end'])
df_ventas_clean['report_generated_on'] = pd.to_datetime(df_ventas_clean['report_generated_on'])

# Mostrar la fila problemática antes de eliminarla
fila_anomala = df_ventas_clean[df_ventas_clean['amount'] <= 0]
print(f"Filas con amount <= 0: {len(fila_anomala)}")
display(fila_anomala)

# Eliminar filas con amount <= 0: units > 0 y amount <= 0 implica precio nulo o negativo
df_ventas_clean = df_ventas_clean[df_ventas_clean['amount'] > 0]

print(f"Filas antes: {len(df_ventas)} | después: {len(df_ventas_clean)}")

Filas con amount <= 0: 26


,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,department_code,department_name,article_code,article_name,units,amount
88,01-05-10.xls,2025-10-01,2025-10-05,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,13,POSTRES,2203,CAFE DOBLE,1.0,-0.00003
192,01-07-06.xls,2026-06-01,2026-06-07,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,1703,NORDÉS,1.0,0.00000
194,01-07-06.xls,2026-06-01,2026-06-07,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,1706,LICORES VARIOS,3.0,0.00000
203,01-07-06.xls,2026-06-01,2026-06-07,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,2155,BAILEYS,2.0,-0.00001
370,01-07-12.xls,2025-12-01,2025-12-07,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,13,POSTRES,1923,SORBETE,3.0,0.00000
1021,05-11-01.xls,2026-01-05,2026-01-11,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,1706,LICORES VARIOS,1.0,0.00000
1143,06-09-07.xls,2026-07-06,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,1989,LICOR PREMIUM,4.0,0.00000
1172,06-09-07.xls,2026-07-06,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,15,OTROS,2154,PAN SIN GLUTEN,1.0,0.00000
1388,06-12-10.xls,2025-10-06,2025-10-12,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,1706,LICORES VARIOS,1.0,-0.00001
1531,08-14-06.xls,2026-06-08,2026-06-14,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,6,LICORES,2155,BAILEYS,2.0,0.00000


Filas antes: 5532 | después: 5506


### 2.3 Reservas

In [24]:
# ── 2.3 RESERVAS ─────────────────────────────────────────────────────────────
df_reservas_clean = df_reservas.copy()

# A) Eliminar columnas residuales con >70% nulos sin valor para el modelo
#   - `reference` (99.99%)
#   - `entered_by` (85.6%)
#   - `referrer` (70.2%).
cols_eliminar = ['reference', 'entered_by', 'referrer']
df_reservas_clean = df_reservas_clean.drop(columns=cols_eliminar)
print(f"Columnas eliminadas: {cols_eliminar}")
print(f"Columnas restantes: {df_reservas_clean.shape[1]}")

# B) Normalizar 'origin': unificar variantes del mismo canal
print("\nOrigin antes:")
print(df_reservas_clean['origin'].value_counts(dropna=False))
df_reservas_clean['origin'] = df_reservas_clean['origin'].replace({'app-movil': 'appmovil'})
print("\nOrigin después:")
print(df_reservas_clean['origin'].value_counts(dropna=False))

# C) Agrupar 'status' en categorías operativas
# Se conserva 'status' original y se crea 'status_agrupado'
STATUS_MAP = {
    'Sentada':                     'completada',
    'Llegada':                     'completada',
    'Cuenta solicitada':           'completada',
    'Liberada':                    'completada',
    'No show':                     'no_show',
    'Cancelado por el cliente':    'cancelada',
    'Cancelado por el restaurante':'cancelada',
    'Confirmada':                  'pendiente',
    'Reconfirmada':                'pendiente',
    'Pendiente':                   'pendiente',
    'A revisar':                   'indefinido',
}
df_reservas_clean['status_agrupado'] = df_reservas_clean['status'].map(STATUS_MAP)

# Verificar que no queda ningún status sin mapear
sin_mapear = df_reservas_clean['status_agrupado'].isna().sum()
print(f"\nStatus sin mapear: {sin_mapear}")
print(df_reservas_clean['status_agrupado'].value_counts(dropna=False))

# D) Excluir reservas con fecha posterior al último ticket disponible
FECHA_CORTE = pd.Timestamp('2026-07-09')
n_futuras = (df_reservas_clean['reservation_datetime'] > FECHA_CORTE).sum()
print(f"\nReservas con fecha futura (>{FECHA_CORTE.date()}): {n_futuras}")
df_reservas_clean = df_reservas_clean[df_reservas_clean['reservation_datetime'] <= FECHA_CORTE]

# E) Documentar walk-ins (created > reservation es comportamiento esperado, no un error)
df_reservas_clean['es_walk_in'] = df_reservas_clean['origin'] == 'walk in'
walk_ins_incoherentes = (
    (df_reservas_clean['created_datetime'] > df_reservas_clean['reservation_datetime']) &
    ~df_reservas_clean['es_walk_in']
).sum()
print(f"\nRegistros con created > reservation que NO son walk-in: {walk_ins_incoherentes}")
# Si el resultado es 0, la incoherencia está 100% explicada por walk-ins

print(f"\nFilas antes: {len(df_reservas)} | después: {len(df_reservas_clean)}")

Columnas eliminadas: ['reference', 'entered_by', 'referrer']
Columnas restantes: 17

Origin antes:
origin
software     6429
terceros     6365
moduloweb    5271
walk in      3072
appmovil      139
app-movil      65
Name: count, dtype: int64

Origin después:
origin
software     6429
terceros     6365
moduloweb    5271
walk in      3072
appmovil      204
Name: count, dtype: int64

Status sin mapear: 0
status_agrupado
completada    17790
cancelada      2893
no_show         493
pendiente       162
indefinido        3
Name: count, dtype: int64

Reservas con fecha futura (>2026-07-09): 52

Registros con created > reservation que NO son walk-in: 38

Filas antes: 21341 | después: 21289


### 2.4 Tips

La suma `document_amount + tip == document_total` se cumple en el 100% de los
registros: coherencia interna perfecta. Un único registro tiene `tip >
document_amount`, lo que es anómalo pero no imposible. Se flaGuea, no se elimina.

In [25]:
df_tips_clean = df_tips.copy()

# Verificar coherencia interna: document_amount + tip == document_total
df_tips_clean['_check'] = (df_tips_clean['document_amount'] + df_tips_clean['tip']).round(2)
diff = (df_tips_clean['_check'] - df_tips_clean['document_total'].round(2)).abs()
print(f"Filas con incoherencia amount + tip != total (diff > 0.01€): {(diff > 0.01).sum()}")
df_tips_clean = df_tips_clean.drop(columns=['_check'])

# Flag tip anómala: propina superior al importe del pedido
df_tips_clean['flag_tip_anomala'] = df_tips_clean['tip'] > df_tips_clean['document_amount']
print(f"Tips anómalas (tip > document_amount): {df_tips_clean['flag_tip_anomala'].sum()}")
display(df_tips_clean[df_tips_clean['flag_tip_anomala']])

print(f"Filas antes: {len(df_tips)} | después: {len(df_tips_clean)}")

Filas con incoherencia amount + tip != total (diff > 0.01€): 0
Tips anómalas (tip > document_amount): 1


,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,document_id,document_amount,tip,document_total,flag_tip_anomala
1263,PROPINAS_TICKET.xls,2025-01-01,2026-07-07,2026-07-07,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,00001TM00005062,2.95,5.0,7.95,True


Filas antes: 1697 | después: 1697


### 2.5 Artículos y departamentos

`article_short_name` (95.6% nulos) y `department_short_name` (100% nulos)
son campos residuales del TPV sin utilidad para el modelo. Se eliminan.
Los códigos de departamento en artículos están perfectamente alineados con
el maestro de departamentos: 0 referencias huérfanas.

In [26]:
# Artículos: eliminar article_short_name (95.6% nulos, sin valor operativo)
df_artic_clean = df_artic.drop(columns=['article_short_name'])
print(f"Artículos — columnas antes: {df_artic.shape[1]} | después: {df_artic_clean.shape[1]}")

# Departamentos: eliminar department_short_name (100% nulos)
df_depart_clean = df_depart.drop(columns=['department_short_name'])
print(f"Departamentos — columnas antes: {df_depart.shape[1]} | después: {df_depart_clean.shape[1]}")

# Verificación cruzada: todos los department_code en artículos existen en departamentos
dept_en_artic  = set(df_artic_clean['department_code'].dropna().unique())
dept_en_depart = set(df_depart_clean['department_code'].dropna().unique())
huerfanos = dept_en_artic - dept_en_depart
print(f"\nDepartment codes en artículos no presentes en maestro: {huerfanos if huerfanos else 'Ninguno ✓'}")

# Sin cambios en menú
df_menu_clean = df_menu.copy()

Artículos — columnas antes: 5 | después: 4
Departamentos — columnas antes: 4 | después: 3

Department codes en artículos no presentes en maestro: Ninguno ✓


### 2.6 Meteorología 

Sin nulos ni outliers físicamente inverosímiles. Se renombran las columnas eliminando las unidades embebidas en el nombre, incompatibles con la mayoría de librerías de ML. `sunshine_duration` se convierte de segundos a horas.

In [27]:
meteo_diaria_clean = meteo_diaria.copy()

# Verificar nombres exactos antes de renombrar
print("Columnas actuales:")
print(meteo_diaria_clean.columns.tolist())

# Renombrar eliminando unidades embebidas en el nombre
RENAME_METEO = {c: c for c in meteo_diaria_clean.columns}  # placeholder
# Sustituye este diccionario con los nombres exactos que imprima el print anterior:
RENAME_METEO = {
    'temperature_2m_max (°C)':    'temperature_max',
    'temperature_2m_min (°C)':    'temperature_min',
    'temperature_2m_mean (°C)':   'temperature_mean',
    'precipitation_sum (mm)':     'precipitation_mm',
    'rain_sum (mm)':              'rain_mm',
    'precipitation_hours (h)':    'precipitation_hours',
    'wind_speed_10m_max (km/h)':  'wind_speed_max',
    'sunshine_duration (s)':      'sunshine_duration_s',
}
meteo_diaria_clean = meteo_diaria_clean.rename(columns=RENAME_METEO)

# Convertir sunshine_duration de segundos a horas
meteo_diaria_clean['sunshine_duration_h'] = (
    meteo_diaria_clean['sunshine_duration_s'] / 3600
)

# Verificar rangos físicamente plausibles para Madrid
print(f"\nTemperatura max: {meteo_diaria_clean['temperature_max'].min():.1f} → {meteo_diaria_clean['temperature_max'].max():.1f} °C")
print(f"Temperatura min: {meteo_diaria_clean['temperature_min'].min():.1f} → {meteo_diaria_clean['temperature_min'].max():.1f} °C")
print(f"Precipitación:   {meteo_diaria_clean['precipitation_mm'].min():.1f} → {meteo_diaria_clean['precipitation_mm'].max():.1f} mm")
print(f"Viento max:      {meteo_diaria_clean['wind_speed_max'].min():.1f} → {meteo_diaria_clean['wind_speed_max'].max():.1f} km/h")

Columnas actuales:
['date', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)', 'temperature_2m_mean (°C)', 'precipitation_sum (mm)', 'rain_sum (mm)', 'precipitation_hours (h)', 'wind_speed_10m_max (km/h)', 'sunshine_duration (s)']

Temperatura max: 3.8 → 40.1 °C
Temperatura min: -4.1 → 25.3 °C
Precipitación:   0.0 → 34.2 mm
Viento max:      4.5 → 38.0 km/h


### 2.7 Festivos y eventos

Datasets externos sin problemas de calidad. Solo se aseguran los tipos datetime
y se verifica que la cobertura temporal engloba el período de tickets.

In [28]:
df_festivos_clean = df_festivos.copy()
df_festivos_clean['fecha'] = pd.to_datetime(df_festivos_clean['fecha'])

# Verificar cobertura respecto al período de tickets
print(f"Festivos — rango: {df_festivos_clean['fecha'].min().date()} → {df_festivos_clean['fecha'].max().date()}")
print(f"es_festivo valores únicos: {df_festivos_clean['es_festivo'].unique()}")

df_eventos_clean = df_eventos.copy()
df_eventos_clean['fecha_inicio'] = pd.to_datetime(df_eventos_clean['fecha_inicio'])
df_eventos_clean['fecha_fin']    = pd.to_datetime(df_eventos_clean['fecha_fin'])

# Verificar coherencia temporal
incoherentes = (df_eventos_clean['fecha_inicio'] > df_eventos_clean['fecha_fin']).sum()
print(f"\nEventos con fecha_inicio > fecha_fin: {incoherentes}")
print(f"Eventos — rango: {df_eventos_clean['fecha_inicio'].min().date()} → {df_eventos_clean['fecha_fin'].max().date()}")

# total_articles: sin cambios, se conserva como está
total_articles_clean = total_articles.copy()

Festivos — rango: 2025-01-01 → 2026-12-25
es_festivo valores únicos: [1]

Eventos con fecha_inicio > fecha_fin: 0
Eventos — rango: 2025-10-10 → 2026-10-01
